# Structured Topic Drift Index (STDI) Audit

This notebook audits rewritten news using the Structured Topic Drift Index (STDI).

Flow:
1) Loads rewritten datasets
2) Extracts structured topic representations for original and rewritten texts
3) Computes STDI and component drifts
4) Saves per-dataset and consolidated outputs

## 1) Imports

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd

from enums import Models, Provider

## 2) Configuration

In [ ]:
# Folder with rewritten CSVs exported by the simulation notebook
run_id_env = os.getenv("RUN_ID")
if run_id_env:
    RUN_ID = run_id_env
    INPUT_DIR = Path("../output/runs") / RUN_ID / "rewritten"
    AUDIT_OUTPUT_DIR = Path("../output/runs") / RUN_ID / "audit" / "TopicDriftAudit"
else:
    RUN_ID = None
    INPUT_DIR = Path("../output/rewritten")
    AUDIT_OUTPUT_DIR = Path("../output/audit/TopicDriftAudit")

# Global dataset selection:
# - "ALL" to process all CSVs in INPUT_DIR
# - file name (e.g.: "science_rewritten_df.csv") to process a single file
DATASET_SELECTOR = "ALL"

# Text columns
ORIGINAL_COLUMN = "description"
REWRITTEN_COLUMN = "rewritten_news"
TITLE_COLUMN = "title"
ROW_ID_COLUMN = None  # optional, e.g.: "id"

# STDI model configuration
TOPIC_DRIFT_PROVIDER = Provider.GEMINI
TOPIC_DRIFT_MODEL = Models.GEMINI31FlashLite
TOPIC_DRIFT_API_KEY = None
TOPIC_DRIFT_BASE_URL = None

# Request pacing
MAX_REQUESTS_PER_MINUTE = 10
RETRY_ATTEMPTS = 2
SLEEP_SECONDS = 0.0

# Analysis scope
MAX_ROWS_PER_DATASET = 200
HIGH_DRIFT_THRESHOLD = 0.60

# Output folder
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
if RUN_ID:
    print(f"RUN_ID: {RUN_ID}")
else:
    print("RUN_ID not set. Using default folders in ../output.")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"DATASET_SELECTOR: {DATASET_SELECTOR}")
print(f"ORIGINAL_COLUMN: {ORIGINAL_COLUMN}")
print(f"REWRITTEN_COLUMN: {REWRITTEN_COLUMN}")
print(f"TOPIC_DRIFT_PROVIDER: {TOPIC_DRIFT_PROVIDER}")
print(f"TOPIC_DRIFT_MODEL: {TOPIC_DRIFT_MODEL}")
print(f"AUDIT_OUTPUT_DIR: {AUDIT_OUTPUT_DIR}")

## 3) Utility functions

In [ ]:
from utils.bert_audit_functions import read_dataset, validate_pair_columns
from utils.run_report import append_execution_report, resolve_execution_report_path
from utils.topic_drift_functions import annotate_stdi_for_rewrites

## 4) Load paired data

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")

if DATASET_SELECTOR.strip().upper() == "ALL":
    selected_files = sorted(INPUT_DIR.glob("*.csv"), key=lambda p: p.name.lower())
else:
    selected_file = INPUT_DIR / DATASET_SELECTOR
    if not selected_file.exists():
        raise FileNotFoundError(f"Selected file not found: {selected_file}")
    selected_files = [selected_file]

if not selected_files:
    raise FileNotFoundError(f"No CSV found in: {INPUT_DIR}")

dataset_payloads = []

for file_path in selected_files:
    df = read_dataset(file_path).copy()
    validate_pair_columns(df, ORIGINAL_COLUMN, REWRITTEN_COLUMN)

    df[ORIGINAL_COLUMN] = df[ORIGINAL_COLUMN].fillna("").astype(str).str.strip()
    df[REWRITTEN_COLUMN] = df[REWRITTEN_COLUMN].fillna("").astype(str).str.strip()
    df = df[(df[ORIGINAL_COLUMN] != "") & (df[REWRITTEN_COLUMN] != "")].copy()
    df = df.head(MAX_ROWS_PER_DATASET).copy()

    dataset_payloads.append(
        {
            "file_path": file_path,
            "file_name": file_path.name,
            "dataset_name": file_path.stem,
            "paired_df": df,
        }
    )

print(f"Selected datasets: {len(dataset_payloads)}")
for payload in dataset_payloads:
    print(f"- {payload['file_name']}: {len(payload['paired_df'])} pairs")

dataset_payloads[0]["paired_df"][[ORIGINAL_COLUMN, REWRITTEN_COLUMN]].head()

## 5) Run STDI analysis

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
audit_started_at = time.perf_counter()

all_audits = []
summary_rows = []

for payload in dataset_payloads:
    paired_df = payload["paired_df"].copy()
    stdi_df = annotate_stdi_for_rewrites(
        df=paired_df,
        rewritten_column=REWRITTEN_COLUMN,
        text_column=ORIGINAL_COLUMN,
        title_column=TITLE_COLUMN,
        provider=TOPIC_DRIFT_PROVIDER,
        model=TOPIC_DRIFT_MODEL,
        api_key=TOPIC_DRIFT_API_KEY,
        base_url=TOPIC_DRIFT_BASE_URL,
        max_rows=MAX_ROWS_PER_DATASET,
        sleep_seconds=SLEEP_SECONDS,
        max_requests_per_minute=MAX_REQUESTS_PER_MINUTE,
        retry_attempts=RETRY_ATTEMPTS,
    )
    stdi_df["source_file"] = payload["file_name"]
    stdi_df["dataset_name"] = payload["dataset_name"]
    stdi_df["high_topic_drift_flag"] = (
        stdi_df["rewritten_news_stdi_vs_original"].fillna(0.0) >= HIGH_DRIFT_THRESHOLD
    )

    if ROW_ID_COLUMN and ROW_ID_COLUMN in stdi_df.columns:
        stdi_df["row_id"] = stdi_df[ROW_ID_COLUMN]

    all_audits.append(stdi_df)

    successful_rows = int(stdi_df["rewritten_news_topic_structure_status"].eq("success").sum())
    high_drift_count = int(stdi_df["high_topic_drift_flag"].sum())
    mean_stdi = (
        float(stdi_df["rewritten_news_stdi_vs_original"].dropna().mean())
        if stdi_df["rewritten_news_stdi_vs_original"].notna().any()
        else 0.0
    )
    max_stdi = (
        float(stdi_df["rewritten_news_stdi_vs_original"].dropna().max())
        if stdi_df["rewritten_news_stdi_vs_original"].notna().any()
        else 0.0
    )
    summary_rows.append(
        {
            "dataset_name": payload["dataset_name"],
            "source_file": payload["file_name"],
            "rows": int(len(stdi_df)),
            "rows_with_successful_stdi": successful_rows,
            "high_drift_count": high_drift_count,
            "high_drift_rate": (high_drift_count / successful_rows) if successful_rows > 0 else 0.0,
            "mean_stdi": mean_stdi,
            "max_stdi": max_stdi,
            "mean_theme_drift": float(
                stdi_df["rewritten_news_theme_drift_vs_original"].dropna().mean()
            )
            if stdi_df["rewritten_news_theme_drift_vs_original"].notna().any()
            else 0.0,
            "mean_subtopic_drift": float(
                stdi_df["rewritten_news_subtopic_drift_vs_original"].dropna().mean()
            )
            if stdi_df["rewritten_news_subtopic_drift_vs_original"].notna().any()
            else 0.0,
            "mean_entity_drift": float(
                stdi_df["rewritten_news_entity_drift_vs_original"].dropna().mean()
            )
            if stdi_df["rewritten_news_entity_drift_vs_original"].notna().any()
            else 0.0,
            "mean_relation_drift": float(
                stdi_df["rewritten_news_relation_drift_vs_original"].dropna().mean()
            )
            if stdi_df["rewritten_news_relation_drift_vs_original"].notna().any()
            else 0.0,
        }
    )

audit_all_df = pd.concat(all_audits, ignore_index=True) if all_audits else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows).sort_values("mean_stdi", ascending=False)
audit_elapsed_seconds = time.perf_counter() - audit_started_at

print("STDI summary by dataset:")
display(summary_df)

audit_all_df[
    [
        "dataset_name",
        ORIGINAL_COLUMN,
        REWRITTEN_COLUMN,
        "rewritten_news_stdi_vs_original",
        "rewritten_news_theme_drift_vs_original",
        "rewritten_news_subtopic_drift_vs_original",
        "rewritten_news_entity_drift_vs_original",
        "rewritten_news_relation_drift_vs_original",
        "high_topic_drift_flag",
    ]
].head(10)

## 6) Save results and inspect highest-drift cases

In [ ]:
if audit_all_df.empty:
    raise ValueError("No STDI result was generated.")

per_file_outputs = []
for dataset_name, dataset_audit in audit_all_df.groupby("dataset_name", dropna=False):
    output_file = AUDIT_OUTPUT_DIR / f"{dataset_name}_stdi_audit.csv"
    dataset_audit.to_csv(output_file, index=False)
    per_file_outputs.append(output_file)

combined_output = AUDIT_OUTPUT_DIR / "all_datasets_stdi_audit.csv"
audit_all_df.to_csv(combined_output, index=False)

summary_output = AUDIT_OUTPUT_DIR / "stdi_summary.csv"
summary_df.to_csv(summary_output, index=False)

print("Saved audit files:")
for output_file in per_file_outputs:
    print(f"- {output_file}")
print(f"- {combined_output}")
print(f"- {summary_output}")

run_id_for_report, report_path = resolve_execution_report_path()

audit_dataset_metrics = []
if not summary_df.empty:
    for row in summary_df.to_dict("records"):
        audit_dataset_metrics.append(
            {
                "dataset_name": row.get("dataset_name"),
                "source_file": row.get("source_file"),
                "rows_audited": int(row.get("rows", 0) or 0),
                "rows_with_successful_stdi": int(row.get("rows_with_successful_stdi", 0) or 0),
                "high_drift_count": int(row.get("high_drift_count", 0) or 0),
                "high_drift_rate": float(row.get("high_drift_rate", 0.0) or 0.0),
                "mean_stdi": float(row.get("mean_stdi", 0.0) or 0.0),
                "max_stdi": float(row.get("max_stdi", 0.0) or 0.0),
                "mean_theme_drift": float(row.get("mean_theme_drift", 0.0) or 0.0),
                "mean_subtopic_drift": float(row.get("mean_subtopic_drift", 0.0) or 0.0),
                "mean_entity_drift": float(row.get("mean_entity_drift", 0.0) or 0.0),
                "mean_relation_drift": float(row.get("mean_relation_drift", 0.0) or 0.0),
            }
        )

successful_stdi_total = int(
    audit_all_df["rewritten_news_topic_structure_status"].eq("success").sum()
)
high_drift_total = int(audit_all_df["high_topic_drift_flag"].sum())
high_drift_rate_total = (
    (high_drift_total / successful_stdi_total) if successful_stdi_total > 0 else 0.0
)
mean_stdi_total = (
    float(audit_all_df["rewritten_news_stdi_vs_original"].dropna().mean())
    if audit_all_df["rewritten_news_stdi_vs_original"].notna().any()
    else 0.0
)
mean_theme_drift_total = (
    float(audit_all_df["rewritten_news_theme_drift_vs_original"].dropna().mean())
    if audit_all_df["rewritten_news_theme_drift_vs_original"].notna().any()
    else 0.0
)
mean_subtopic_drift_total = (
    float(audit_all_df["rewritten_news_subtopic_drift_vs_original"].dropna().mean())
    if audit_all_df["rewritten_news_subtopic_drift_vs_original"].notna().any()
    else 0.0
)
mean_entity_drift_total = (
    float(audit_all_df["rewritten_news_entity_drift_vs_original"].dropna().mean())
    if audit_all_df["rewritten_news_entity_drift_vs_original"].notna().any()
    else 0.0
)
mean_relation_drift_total = (
    float(audit_all_df["rewritten_news_relation_drift_vs_original"].dropna().mean())
    if audit_all_df["rewritten_news_relation_drift_vs_original"].notna().any()
    else 0.0
)

append_execution_report(
    report_path=report_path,
    notebook_name="topic_drift_audit_workbench.ipynb",
    section_title="Structured topic drift audit execution",
    run_id=run_id_for_report,
    details={
        "rows_audited": int(len(audit_all_df)),
        "audit_execution_seconds": round(audit_elapsed_seconds, 3),
        "topic_drift_auditor_used": f"{TOPIC_DRIFT_PROVIDER}::{TOPIC_DRIFT_MODEL}",
        "total_dataset_rows": int(sum(len(payload["paired_df"]) for payload in dataset_payloads)),
        "audit_model_metrics": [
            {
                "audit_model": f"{TOPIC_DRIFT_PROVIDER}::{TOPIC_DRIFT_MODEL}",
                "successful_stdi_rows": successful_stdi_total,
                "high_drift_count": high_drift_total,
                "high_drift_rate": round(high_drift_rate_total, 6),
                "mean_stdi": round(mean_stdi_total, 6),
                "mean_theme_drift": round(mean_theme_drift_total, 6),
                "mean_subtopic_drift": round(mean_subtopic_drift_total, 6),
                "mean_entity_drift": round(mean_entity_drift_total, 6),
                "mean_relation_drift": round(mean_relation_drift_total, 6),
            }
        ],
        "audit_dataset_metrics": audit_dataset_metrics,
        "execution_details": {
            "dataset_selector": DATASET_SELECTOR,
            "input_dir": str(INPUT_DIR),
            "original_column": ORIGINAL_COLUMN,
            "rewritten_column": REWRITTEN_COLUMN,
            "high_drift_threshold": HIGH_DRIFT_THRESHOLD,
            "max_requests_per_minute": MAX_REQUESTS_PER_MINUTE,
        },
    },
)

print(f"Execution report updated: {report_path}")

highest_drift_cases = audit_all_df.sort_values(
    "rewritten_news_stdi_vs_original", ascending=False
).head(20)
highest_drift_cases[
    [
        "dataset_name",
        ORIGINAL_COLUMN,
        REWRITTEN_COLUMN,
        "rewritten_news_stdi_vs_original",
        "rewritten_news_theme_drift_vs_original",
        "rewritten_news_subtopic_drift_vs_original",
        "rewritten_news_entity_drift_vs_original",
        "rewritten_news_relation_drift_vs_original",
        "high_topic_drift_flag",
    ]
]